In [1]:
# Cell 1: Setup (Run this FIRST before any other code)
import os
import sys

from pathlib import Path

# Add src to Python path
src_path = Path.cwd() / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    print(f"✅ Added {src_path} to sys.path")

# Verify
print(f"✅ Python path updated")
print(f"✅ Working directory: {Path.cwd()}")

✅ Added c:\Users\Dell\OneDrive\Desktop\mlops-project\chest-ct-ecs\research\src to sys.path
✅ Python path updated
✅ Working directory: c:\Users\Dell\OneDrive\Desktop\mlops-project\chest-ct-ecs\research


In [2]:
%pwd

'c:\\Users\\Dell\\OneDrive\\Desktop\\mlops-project\\chest-ct-ecs\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Dell\\OneDrive\\Desktop\\mlops-project\\chest-ct-ecs'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
# Auto-fix common.py
from pathlib import Path

common_py_path = Path("src/cnnClassifier/utils/common.py")

# Read existing content
with open(common_py_path, 'r') as f:
    content = f.read()

# Add S3 functions if not present
if 'def download_from_s3' not in content:
    s3_functions = '''

# ============================================
# S3 FUNCTIONS
# ============================================

def download_from_s3(s3_uri: str, local_path: str) -> bool:
    """Download file from S3 bucket"""
    import boto3
    import os
    
    try:
        s3_path = s3_uri.replace('s3://', '')
        bucket_name, key = s3_path.split('/', 1)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3_client = boto3.client('s3')
        s3_client.download_file(bucket_name, key, local_path)
        logger.info(f"Downloaded from s3://{bucket_name}/{key}")
        return True
    except Exception as e:
        logger.error(f"S3 download failed: {e}")
        raise e

def extract_zip_file(zip_path: str, extract_to: str):
    """Extract zip file"""
    import zipfile
    import os
    
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        logger.info(f"Extracted {len(zip_ref.namelist())} files")
'''
    
    with open(common_py_path, 'a') as f:
        f.write(s3_functions)
    
    print("✅ Added S3 functions to common.py")
else:
    print("✅ S3 functions already present")

print("Ready to import!")

✅ S3 functions already present
Ready to import!


In [7]:
# CELL 1: Fix common.py (run this first)
from pathlib import Path

common_py_path = Path("src/cnnClassifier/utils/common.py")
fixed_content = '''import os
import yaml
import json
import joblib
import base64
import boto3
import zipfile
from pathlib import Path
from typing import Any
from box import ConfigBox
from box.exceptions import BoxValueError
from ensure import ensure_annotations
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def read_yaml(path_to_yaml):
    with open(path_to_yaml) as yaml_file:
        return ConfigBox(yaml.safe_load(yaml_file))

def create_directories(paths, verbose=True):
    for path in paths:
        os.makedirs(path, exist_ok=True)

def download_from_s3(s3_uri, local_path):
    s3_path = s3_uri.replace('s3://', '')
    bucket, key = s3_path.split('/', 1)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    boto3.client('s3').download_file(bucket, key, local_path)
    logger.info(f"Downloaded: {s3_uri}")

def extract_zip_file(zip_path, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        logger.info(f"Extracted {len(zip_ref.namelist())} files")
'''
common_py_path.write_text(fixed_content)
print("✅ Fixed common.py")

✅ Fixed common.py


In [8]:
# Add src to path
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

# Now imports should work
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

print("✅ Success! All imports working!")


✅ Success! All imports working!


In [9]:
# First, define your constants
from pathlib import Path

# Define the paths
CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")

print(f"✅ CONFIG_FILE_PATH: {CONFIG_FILE_PATH}")
print(f"✅ PARAMS_FILE_PATH: {PARAMS_FILE_PATH}")

# Check if files exist
print(f"Config exists: {CONFIG_FILE_PATH.exists()}")
print(f"Params exists: {PARAMS_FILE_PATH.exists()}")

✅ CONFIG_FILE_PATH: config\config.yaml
✅ PARAMS_FILE_PATH: params.yaml
Config exists: True
Params exists: True


In [10]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self):
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        
        return {
            "root_dir": config.root_dir,
            "source_URL": config.source_URL,
            "local_data_file": config.local_data_file,
            "unzip_dir": config.unzip_dir
        }

# Use it
config_manager = ConfigurationManager()
data_config = config_manager.get_data_ingestion_config()

print("✅ Working without DataIngestionConfig import!")
print(f"Root dir: {data_config['root_dir']}")
print(f"Source URL: {data_config['source_URL']}")

✅ Working without DataIngestionConfig import!
Root dir: artifacts/data_ingestion
Source URL: s3://chest-ct-raw-data/chest-data.zip


In [11]:
from pathlib import Path

common_file = Path("src/cnnClassifier/utils/common.py")

# Read current file
content = common_file.read_text()

# Check if get_size exists
if 'def get_size' not in content:
    # Add get_size function at the end of the file
    get_size_function = '''

def get_size(path):
    """Get file size in human readable format"""
    import os
    size_bytes = os.path.getsize(path)
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} TB"
'''
    
    # Append to file
    with open(common_file, 'a') as f:
        f.write(get_size_function)
    print("✅ Added get_size function to common.py")
else:
    print("✅ get_size function already exists")

# Verify
print("\nVerifying get_size is now available...")
import importlib.util
spec = importlib.util.spec_from_file_location("common", common_file)
common = importlib.util.module_from_spec(spec)
spec.loader.exec_module(common)
print(f"✅ Functions available: {[f for f in dir(common) if f.startswith('get_')]}")

✅ Added get_size function to common.py

Verifying get_size is now available...
✅ Functions available: ['get_size']


In [12]:
from pathlib import Path

common_file = Path("src/cnnClassifier/utils/common.py")

# Complete working common.py
complete_content = '''import os
import yaml
import json
import joblib
import base64
import boto3
import zipfile
from pathlib import Path
from typing import Any
from box import ConfigBox
from box.exceptions import BoxValueError
from ensure import ensure_annotations
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def get_size(path):
    """Get file size in human readable format"""
    import os
    size_bytes = os.path.getsize(path)
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} TB"


@ensure_annotations
def read_yaml(path_to_yaml: Path) -> ConfigBox:
    """Reads yaml file and returns ConfigBox type"""
    try:
        with open(path_to_yaml) as yaml_file:
            content = yaml.safe_load(yaml_file)
            logger.info(f"yaml file: {path_to_yaml} loaded successfully")
            return ConfigBox(content)
    except BoxValueError:
        raise ValueError("yaml file is empty")
    except Exception as e:
        raise e


@ensure_annotations
def create_directories(path_to_directories: list, verbose=True):
    """Create list of directories"""
    for path in path_to_directories:
        os.makedirs(path, exist_ok=True)
        if verbose:
            logger.info(f"created directory at: {path}")


@ensure_annotations
def save_json(path: Path, data: dict):
    """Save json data"""
    with open(path, "w") as f:
        json.dump(data, f, indent=4)
    logger.info(f"json file saved at: {path}")


@ensure_annotations
def load_json(path: Path) -> ConfigBox:
    """Load json files data"""
    with open(path) as f:
        content = json.load(f)
    logger.info(f"json file loaded successfully from: {path}")
    return ConfigBox(content)


@ensure_annotations
def save_bin(data: Any, path: Path):
    """Save binary file"""
    joblib.dump(value=data, filename=path)
    logger.info(f"binary file saved at: {path}")


@ensure_annotations
def load_bin(path: Path) -> Any:
    """Load binary data"""
    data = joblib.load(path)
    logger.info(f"binary file loaded from: {path}")
    return data


def decodeImage(imgstring, fileName):
    """Decode base64 image and save to file"""
    imgdata = base64.b64decode(imgstring)
    with open(fileName, 'wb') as f:
        f.write(imgdata)
        f.close()


def encodeImageIntoBase64(croppedImagePath):
    """Encode image to base64"""
    with open(croppedImagePath, "rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')


def download_from_s3(s3_uri: str, local_path: str) -> bool:
    """Download file from S3 bucket"""
    try:
        s3_path = s3_uri.replace('s3://', '')
        bucket, key = s3_path.split('/', 1)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3_client = boto3.client('s3')
        s3_client.download_file(bucket, key, local_path)
        logger.info(f"Downloaded from s3://{bucket}/{key} to {local_path}")
        return True
    except Exception as e:
        logger.error(f"S3 download failed: {e}")
        raise e


def extract_zip_file(zip_path: str, extract_to: str):
    """Extract zip file to destination directory"""
    try:
        os.makedirs(extract_to, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
            file_count = len(zip_ref.namelist())
            logger.info(f"Extracted {file_count} files to {extract_to}")
    except Exception as e:
        logger.error(f"Extraction failed: {e}")
        raise e
'''

# Write the file
common_file.write_text(complete_content)
print(f"✅ Complete common.py written to {common_file}")
print(f"File size: {common_file.stat().st_size} bytes")

✅ Complete common.py written to src\cnnClassifier\utils\common.py
File size: 3714 bytes


In [13]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd() / "src"))

# Remove cached module
if 'cnnClassifier.utils.common' in sys.modules:
    del sys.modules['cnnClassifier.utils.common']

# Import now
from cnnClassifier.utils.common import get_size, download_from_s3, extract_zip_file

print("✅ All functions imported successfully!")
print(f"get_size: {get_size}")
print(f"download_from_s3: {download_from_s3}")
print(f"extract_zip_file: {extract_zip_file}")

# Test get_size
test_file = Path("src/cnnClassifier/utils/common.py")
print(f"\nTest get_size: {get_size(test_file)}")

✅ All functions imported successfully!
get_size: <function get_size at 0x000001B82B63C4A0>
download_from_s3: <function download_from_s3 at 0x000001B82B63CCC0>
extract_zip_file: <function extract_zip_file at 0x000001B82B63CD60>

Test get_size: 3.63 KB


In [14]:
import os
import zipfile
import boto3
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size, download_from_s3, extract_zip_file

print("✅ All imports successful!")
print(f"get_size: {get_size}")
print(f"download_from_s3: {download_from_s3}")
print(f"extract_zip_file: {extract_zip_file}")

✅ All imports successful!
get_size: <function get_size at 0x000001B82B63C4A0>
download_from_s3: <function download_from_s3 at 0x000001B82B63CCC0>
extract_zip_file: <function extract_zip_file at 0x000001B82B63CD60>


In [17]:
class DataIngestion:
    def __init__(self, config):
        self.config = config
    
    def download_file(self):
        try:
            # Use dictionary access with ['key'] instead of .key
            dataset_url = self.config['source_URL']
            zip_download_dir = self.config['local_data_file']
            
            os.makedirs(os.path.dirname(zip_download_dir), exist_ok=True)
            
            logger.info(f"📥 Downloading data from {dataset_url}")
            logger.info(f"📁 Saving to: {zip_download_dir}")
            
            download_from_s3(dataset_url, zip_download_dir)
            
            file_size = get_size(zip_download_dir)
            logger.info(f"✅ Downloaded successfully! File size: {file_size}")
            
        except Exception as e:
            logger.error(f"❌ Download failed: {e}")
            raise e
    
    def extract_zip_file(self):
        try:
            # Use dictionary access with ['key']
            unzip_path = self.config['unzip_dir']
            os.makedirs(unzip_path, exist_ok=True)
            
            with zipfile.ZipFile(self.config['local_data_file'], 'r') as zip_ref:
                zip_ref.extractall(unzip_path)
                file_count = len(zip_ref.namelist())
            
            logger.info(f"✅ Extracted {file_count} files to {unzip_path}")
            
        except Exception as e:
            logger.error(f"❌ Extraction failed: {e}")
            raise e

In [18]:
try:
    config_manager = ConfigurationManager()
    data_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(data_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    print("✅ Data ingestion complete!")
except Exception as e:
    print(f"❌ Error: {e}")

[2026-05-08 13:12:14,811: INFO: 512079300: 📥 Downloading data from s3://chest-ct-raw-data/chest-data.zip]
[2026-05-08 13:12:14,812: INFO: 512079300: 📁 Saving to: artifacts/data_ingestion/data.zip]
[2026-05-08 13:12:14,886: INFO: credentials: Found credentials in shared credentials file: ~/.aws/credentials]
[2026-05-08 13:12:27,125: INFO: common: Downloaded from s3://chest-ct-raw-data/chest-data.zip to artifacts/data_ingestion/data.zip]
[2026-05-08 13:12:27,134: INFO: 512079300: ✅ Downloaded successfully! File size: 46.69 MB]
[2026-05-08 13:12:27,963: INFO: 512079300: ✅ Extracted 346 files to artifacts/data_ingestion]
✅ Data ingestion complete!
